In [1]:
import operator
import warnings
from typing import *
import traceback

import os
import torch
from dotenv import load_dotenv
from IPython.display import Image
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from transformers import logging
import matplotlib.pyplot as plt
import numpy as np
import re

from medrax.agent import *
from medrax.tools import *
from medrax.utils import *

import json
import openai
import os
import glob
import time
import logging
from datetime import datetime
try:
    from experiments.evaluation_utils import extract_answer_letter
except ImportError:
    from evaluation_utils import extract_answer_letter

warnings.filterwarnings("ignore")
_ = load_dotenv()


# Setup directory paths
ROOT = "set this directory to where MedRAX is, .e.g /home/MedRAX"
PROMPT_FILE = f"{ROOT}/medrax/docs/system_prompts.txt"
BENCHMARK_FILE = f"{ROOT}/benchmark/questions"
BENCHMARK_JSONL = f"{ROOT}/chestagentbench/metadata.jsonl"
MODEL_DIR = f"set this to where the tool models are, e.g /home/models"
FIGURES_DIR = f"{ROOT}/benchmark/figures"

model_name = "medrax"
temperature = 0.2
medrax_logs = f"{ROOT}/experiments/medrax_logs"
log_filename = f"{medrax_logs}/{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
logging.basicConfig(filename=log_filename, level=logging.INFO, format="%(message)s", force=True)
device = "cuda"

In [2]:
def get_tools():
    report_tool = ChestXRayReportGeneratorTool(cache_dir=MODEL_DIR, device=device)
    xray_classification_tool = ChestXRayClassifierTool(device=device)
    segmentation_tool = ChestXRaySegmentationTool(device=device)
    grounding_tool = XRayPhraseGroundingTool(
        cache_dir=MODEL_DIR, temp_dir="temp", device=device, load_in_8bit=True
    )
    xray_vqa_tool = XRayVQATool(cache_dir=MODEL_DIR, device=device)
    llava_med_tool = LlavaMedTool(cache_dir=MODEL_DIR, device=device, load_in_8bit=True)

    return [
        report_tool,
        xray_classification_tool,
        segmentation_tool,
        grounding_tool,
        xray_vqa_tool,
        llava_med_tool,
    ]


def get_agent(tools):
    prompts = load_prompts_from_file(PROMPT_FILE)
    prompt = prompts["MEDICAL_ASSISTANT"]

    checkpointer = MemorySaver()
    model = ChatOpenAI(model="gpt-4o", temperature=temperature, top_p=0.95)
    agent = Agent(
        model,
        tools=tools,
        log_tools=True,
        log_dir="logs",
        system_prompt=prompt,
        checkpointer=checkpointer,
    )
    thread = {"configurable": {"thread_id": "1"}}
    return agent, thread


def run_medrax(agent, thread, prompt, image_urls=[]):
    messages = [
        HumanMessage(
            content=[
                {"type": "text", "text": prompt},
            ]
            + [{"type": "image_url", "image_url": {"url": image_url}} for image_url in image_urls]
        )
    ]

    final_response = None
    for event in agent.workflow.stream({"messages": messages}, thread):
        for v in event.values():
            final_response = v

    final_response = final_response["messages"][-1].content.strip()
    agent_state = agent.workflow.get_state(thread)

    return final_response, str(agent_state)

In [3]:
def _flatten(values):
    if not values:
        return []
    if isinstance(values, str):
        return [values]
    return [item for value in values for item in (value if isinstance(value, list) else [value])]


def resolve_question_assets(question_data, case_details, case_id):
    if "images" in question_data:
        image_paths = [
            os.path.join(ROOT, "benchmark", image_path)
            for image_path in _flatten(question_data.get("images", []))
        ]
        figure_prompt = "".join(
            f"{os.path.basename(image_path)} located at {image_path}\n"
            for image_path in image_paths
        )
        return figure_prompt, _flatten(question_data.get("image_source_urls", [])), [], image_paths

    try:
        if isinstance(question_data["figures"], str):
            try:
                required_figures = json.loads(question_data["figures"])
            except json.JSONDecodeError:
                required_figures = [question_data["figures"]]
        elif isinstance(question_data["figures"], list):
            required_figures = question_data["figures"]
        else:
            required_figures = [str(question_data["figures"])]
    except Exception as error:
        print(f"Error parsing figures: {error}")
        required_figures = []

    required_figures = [
        figure if figure.startswith("Figure ") else f"Figure {figure}"
        for figure in required_figures
    ]
    subfigures = []
    for figure in required_figures:
        base_figure_num = "".join(filter(str.isdigit, figure))
        figure_letter = "".join(filter(str.isalpha, figure.split()[-1])) or None
        matching_figures = [
            case_figure
            for case_figure in case_details.get("figures", [])
            if case_figure["number"] == f"Figure {base_figure_num}"
        ]
        if not matching_figures:
            print(f"No matching figure found for {figure} in case {case_id}")
            continue
        for case_figure in matching_figures:
            if figure_letter:
                subfigures.extend(
                    subfig
                    for subfig in case_figure.get("subfigures", [])
                    if subfig.get("number", "").lower().endswith(figure_letter.lower())
                    or subfig.get("label", "").lower() == figure_letter.lower()
                )
            else:
                subfigures.extend(case_figure.get("subfigures", []))

    figure_prompt = ""
    image_urls = []
    image_paths = []
    for subfig in subfigures:
        if "number" in subfig:
            subfig_number = subfig["number"].lower().strip().replace(" ", "_") + ".jpg"
            subfig_path = os.path.join(FIGURES_DIR, case_id, subfig_number)
            image_paths.append(subfig_path)
            figure_prompt += f"{subfig_number} located at {subfig_path}\n"
        if "url" in subfig:
            image_urls.append(subfig["url"])
        else:
            print(f"Subfigure missing URL: {subfig}")
    return figure_prompt, image_urls, subfigures, image_paths


def normalized_metadata(question_data):
    metadata = question_data.get("metadata", {}).copy()
    categories = question_data.get("categories", metadata.get("categories", []))
    if isinstance(categories, str):
        categories = [category.strip() for category in categories.split(",")]
    metadata["categories"] = categories
    return metadata


def create_multimodal_request(question_data, case_details, case_id, question_id, agent, thread):
    start_time = time.time()
    final_response = ""
    raw_model_answer = ""
    model_answer = None
    agent_state = ""
    last_error = None
    attempt_count = 0
    subfigures = []
    image_urls = []
    image_paths = []

    try:
        figure_prompt, image_urls, subfigures, image_paths = resolve_question_assets(
            question_data, case_details, case_id
        )
        prompt = (
            f"Answer this question correctly using chain of thought reasoning and "
            "carefully evaluating choices. Solve using our own vision and reasoning and then"
            "use tools to complement your reasoning. Trust your own judgement over any tools.\n"
            f"{question_data['question']}\n{figure_prompt}"
        )

        for attempt_count in range(1, 4):
            try:
                final_response, agent_state = run_medrax(
                    agent=agent, thread=thread, prompt=prompt, image_urls=image_urls
                )
                raw_model_answer, agent_state = run_medrax(
                    agent=agent,
                    thread=thread,
                    prompt="If you had to choose the best option, only respond with the letter of choice (only one of A, B, C, D, E, F)",
                )
                model_answer = extract_answer_letter(raw_model_answer)
                if model_answer is None:
                    raise ValueError(f"Invalid answer choice: {raw_model_answer!r}")
                last_error = None
                break
            except Exception as error:
                last_error = error
                if attempt_count < 3:
                    print(
                        f"Attempt {attempt_count} failed for case {case_id}, "
                        f"question {question_id}: {error}. Retrying..."
                    )
    except Exception as error:
        last_error = error
        prompt = question_data.get("question", "")

    duration = time.time() - start_time
    log_entry = {
        "case_id": case_id,
        "question_id": question_id,
        "timestamp": datetime.now().isoformat(),
        "model": model_name,
        "temperature": temperature,
        "duration": round(duration, 2),
        "attempts": attempt_count,
        "usage": "",
        "cost": 0,
        "raw_response": final_response,
        "raw_model_answer": raw_model_answer,
        "model_answer": model_answer,
        "correct_answer": extract_answer_letter(question_data.get("answer")),
        "input": {
            "messages": prompt,
            "question_data": {
                "question": question_data.get("question", ""),
                "explanation": question_data.get("explanation", ""),
                "metadata": normalized_metadata(question_data),
                "figures": question_data.get("figures", question_data.get("images", [])),
            },
            "image_urls": image_urls,
            "image_captions": [subfig.get("caption", "") for subfig in subfigures],
            "image_paths": image_paths,
        },
        "agent_state": agent_state,
    }
    if last_error is not None:
        log_entry.update({"status": "error", "error": str(last_error)})
        print(f"Failed case {case_id}, question {question_id} after {attempt_count} attempts: {last_error}")
    else:
        log_entry["status"] = "success"

    logging.info(json.dumps(log_entry))
    return final_response, model_answer


def iter_benchmark_questions():
    if os.path.isfile(BENCHMARK_JSONL):
        with open(BENCHMARK_JSONL, "r") as benchmark_file:
            for line in benchmark_file:
                if line.strip():
                    question_data = json.loads(line)
                    yield (
                        question_data,
                        {},
                        str(question_data["case_id"]),
                        question_data["question_id"],
                    )
        return

    with open(f"{ROOT}/data/eurorad_metadata.json", "r") as metadata_file:
        data = json.load(metadata_file)
    for case_id, case_details in data.items():
        for question_file in glob.glob(f"{BENCHMARK_FILE}/{case_id}/{case_id}_*.json"):
            with open(question_file, "r") as file:
                question_data = json.load(file)
            yield question_data, case_details, case_id, os.path.basename(question_file).split(".")[0]


def main(tools):
    benchmark_questions = list(iter_benchmark_questions())
    if len(benchmark_questions) != 2500:
        raise ValueError(
            f"ChestAgentBench must contain exactly 2500 questions, found {len(benchmark_questions)}"
        )

    total_cases = len({case_id for _, _, case_id, _ in benchmark_questions})
    cases_processed = 0
    questions_processed = 0
    failed_questions = 0
    current_case_id = None
    agent = None
    thread = None

    print(f"Beginning benchmark evaluation for model {model_name} with temperature {temperature}\n")

    for question_data, case_details, case_id, question_id in benchmark_questions:
        if case_id != current_case_id:
            print("----------------------------------------------------------------")
            agent, thread = get_agent(tools)
            current_case_id = case_id
            cases_processed += 1

        questions_processed += 1
        final_response, model_answer = create_multimodal_request(
            question_data, case_details, case_id, question_id, agent, thread
        )
        if model_answer is None:
            failed_questions += 1

        print(
            f"Progress: Case {cases_processed}/{total_cases}, "
            f"Question {questions_processed}/{len(benchmark_questions)}"
        )
        print(f"Case ID: {case_id}")
        print(f"Question ID: {question_id}")
        print(f"Final Response: {final_response}")
        print(f"Model Answer: {model_answer}")
        print(f"Correct Answer: {question_data['answer']}")
        print("----------------------------------------------------------------\n")

    print("\nBenchmark Summary:")
    print(f"Total Cases Processed: {cases_processed}")
    print(f"Total Questions Processed: {questions_processed}")
    print(f"Total Questions Failed: {failed_questions}")


In [ ]:
tools = get_tools()
main(tools)